# Part Proxy — Interactive Visualization

Interactively visualize the **input alpha-wrapped model** and the **fitted
parametric proxy**. Two proxy types are supported and auto-selected per part:

- **generalized_cylinder** (swept tube) — tube-like parts: `spout`, `handle`.
  Layers: centerline, cross-section rings, endpoints.
- **surface_of_revolution** (revolve) — round parts: `teapot_body`, `lid`,
  `knob`. Layers: rotation axis, profile `r(h)`, per-slice rings.

Outputs are organized per part under `output/<part_slug>/`. The files differ by
proxy type (`centerline.ply`/`cross_sections.ply` vs `axis.ply`/`profile.ply`),
and `params.json` carries `proxy_type`.

Set the `PART` variable in the next cell to switch parts (e.g. `spout`,
`handle`, `teapot_body`, `lid`, `knob`).

Use the legend (click entries to toggle), drag to rotate, scroll to zoom.

> Run with the project venv kernel: `Python 3 (.venv)` (created via `uv`).


In [1]:
import json
import os

import numpy as np
import trimesh
import plotly.graph_objects as go

# Resolve repo root whether the notebook runs from repo root or notebooks/.
REPO = os.getcwd()
if os.path.basename(REPO) == 'notebooks':
    REPO = os.path.dirname(REPO)

# ---- Choose which fitted part to visualize ----------------------------- #
# Each part lives in its own folder: output/<part_slug>/{proxy,centerline,...}
PART = 'spout'   # e.g. 'spout', 'knob', 'handle', 'teapot_body', 'lid'

OUTPUT_ROOT = os.path.join(REPO, 'output')
PART_DIR = os.path.join(OUTPUT_ROOT, PART)

# Map each part slug back to its source wrapped mesh (read from params.json).
params_path = os.path.join(PART_DIR, 'params.json')
with open(params_path) as f:
    params = json.load(f)
INPUT_MESH = params['source_mesh_path']
if not os.path.isabs(INPUT_MESH):
    INPUT_MESH = os.path.join(REPO, INPUT_MESH)

available_parts = sorted(
    d for d in os.listdir(OUTPUT_ROOT)
    if os.path.isdir(os.path.join(OUTPUT_ROOT, d))
)
print('Repo:', REPO)
print('Available fitted parts:', available_parts)
print('Selected part:', PART, '->', PART_DIR)
print('Input mesh exists:', os.path.exists(INPUT_MESH))
print('Part files:', sorted(os.listdir(PART_DIR)))

Repo: /work/thy/Mesh2ParametricSurface
Input mesh exists: True
Output dir exists: True
Output files: ['spout_centerline.ply', 'spout_cross_sections.ply', 'spout_debug.png', 'spout_params.json', 'spout_proxy.ply', 'spout_sampled_points.ply']


In [ ]:
# ------------------------------------------------------------------ #
# Loader helpers
# ------------------------------------------------------------------ #
def load_mesh(path):
    m = trimesh.load(path, process=False, force='mesh')
    if isinstance(m, trimesh.Scene):
        m = trimesh.util.concatenate([g for g in m.geometry.values()])
    return m


def load_points(path):
    pc = trimesh.load(path, process=False)
    return np.asarray(pc.vertices)


def load_path_segments(path):
    """Load a Path3D PLY as a list of ordered (M,3) polylines."""
    p = trimesh.load(path, process=False)
    segs = []
    if isinstance(p, trimesh.path.Path3D):
        for ent in p.entities:
            pts = p.vertices[ent.points]
            segs.append(np.asarray(pts))
    else:
        segs.append(np.asarray(p.vertices))
    return segs


def _opt(path, loader, default):
    """Load ``path`` with ``loader`` if it exists, else return ``default``."""
    if os.path.exists(path):
        return loader(path)
    print('  (missing, skipped):', os.path.relpath(path, REPO))
    return default


def _revolve_rings(prm, n_theta=64):
    """Reconstruct per-slice rings of a revolve fit, colored by reliability."""
    ax = prm['axis']
    o = np.array(ax['origin'], float)
    d = np.array(ax['direction'], float)
    u = np.array(ax['u'], float)
    v = np.array(ax['v'], float)
    th = np.linspace(0, 2 * np.pi, n_theta)
    cos_t, sin_t = np.cos(th)[:, None], np.sin(th)[:, None]
    rings, rel = [], []
    for s in prm.get('profile_slices', []):
        ring = o + s['h'] * d + s['r'] * (cos_t * u[None, :] + sin_t * v[None, :])
        rings.append(ring)
        rel.append(s['reliability'])
    return rings, np.array(rel)


def load_part(part):
    """Load all fitted artifacts + the input mesh for one part.

    Works for both proxy types. Returns a dict with a ``proxy_type`` key plus:
    input_mesh, proxy_mesh, sampled_pts, params, and proxy-specific layers:
      * generalized_cylinder: centerline_segs, ring_segs, ep_a, ep_b
      * surface_of_revolution: axis_segs, profile_segs, ring_segs (per slice)
    ``section_reliability`` holds per-ring reliability for either type.
    """
    part_dir = os.path.join(OUTPUT_ROOT, part)
    with open(os.path.join(part_dir, 'params.json')) as f:
        prm = json.load(f)
    proxy_type = prm.get('proxy_type', 'generalized_cylinder')
    input_mesh_path = prm['source_mesh_path']
    if not os.path.isabs(input_mesh_path):
        input_mesh_path = os.path.join(REPO, input_mesh_path)

    def pf(name):
        return os.path.join(part_dir, name)

    d = {
        'part': part,
        'part_dir': part_dir,
        'proxy_type': proxy_type,
        'input_mesh': load_mesh(input_mesh_path),
        'proxy_mesh': _opt(pf('proxy.ply'), load_mesh, trimesh.Trimesh()),
        'sampled_pts': _opt(pf('sampled_points.ply'), load_points, np.zeros((0, 3))),
        'params': prm,
        # Layer defaults shared across proxy types.
        'centerline_segs': [],
        'ring_segs': [],
        'axis_segs': [],
        'profile_segs': [],
        'ep_a': None,
        'ep_b': None,
        'section_reliability': np.zeros(0),
    }

    if proxy_type == 'surface_of_revolution':
        d['axis_segs'] = _opt(pf('axis.ply'), load_path_segments, [])
        d['profile_segs'] = _opt(pf('profile.ply'), load_path_segments, [])
        rings, rel = _revolve_rings(prm)
        d['ring_segs'] = rings
        d['section_reliability'] = rel
    else:
        d['centerline_segs'] = _opt(pf('centerline.ply'), load_path_segments, [])
        d['ring_segs'] = _opt(pf('cross_sections.ply'), load_path_segments, [])
        d['ep_a'] = np.array(prm['endpoints']['a'])
        d['ep_b'] = np.array(prm['endpoints']['b'])
        d['section_reliability'] = np.array(
            [s['reliability'] for s in prm['cross_sections']]
        )
    return d


# Load the selected part and unpack to module-level globals (used by the
# single-part figures below). The all-parts gallery near the bottom calls
# load_part() again for every part.
data = load_part(PART)
input_mesh = data['input_mesh']
proxy_mesh = data['proxy_mesh']
sampled_pts = data['sampled_pts']
centerline_segs = data['centerline_segs']
ring_segs = data['ring_segs']
ep_a = data['ep_a']
ep_b = data['ep_b']
section_reliability = data['section_reliability']
params = data['params']

print('part:', PART, '| proxy_type:', data['proxy_type'])
print('input mesh:', len(input_mesh.vertices), 'verts /', len(input_mesh.faces), 'faces')
print('proxy mesh:', len(proxy_mesh.vertices), 'verts /', len(proxy_mesh.faces), 'faces')
print('sampled points:', sampled_pts.shape)
print('ring segments:', len(ring_segs))
print('coverage_ratio:', params['fitting_metrics'].get('coverage_ratio'))


In [3]:
# ------------------------------------------------------------------ #
# Plotly trace builders
# ------------------------------------------------------------------ #
def mesh_trace(mesh, name, color, opacity=0.5, visible=True):
    v = mesh.vertices
    f = mesh.faces
    return go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color, opacity=opacity, name=name,
        showlegend=True, visible=visible, flatshading=True,
        hoverinfo='name',
    )


def points_trace(pts, name, color, size=1.5, visible=True):
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers', name=name,
        marker=dict(size=size, color=color, opacity=0.5),
        visible=visible, hoverinfo='name',
    )


def polyline_trace(segs, name, color, width=5, visible=True, close=False):
    xs, ys, zs = [], [], []
    for s in segs:
        s = np.asarray(s)
        if close and len(s) > 1:
            s = np.vstack([s, s[:1]])
        xs += list(s[:, 0]) + [None]
        ys += list(s[:, 1]) + [None]
        zs += list(s[:, 2]) + [None]
    return go.Scatter3d(
        x=xs, y=ys, z=zs, mode='lines', name=name,
        line=dict(color=color, width=width),
        visible=visible, hoverinfo='name',
    )


def endpoints_trace(ep_a, ep_b, name='endpoints', visible=True):
    pts = np.vstack([ep_a, ep_b])
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers+text', name=name,
        text=['A', 'B'], textposition='top center',
        marker=dict(size=6, color=['green', 'magenta']),
        visible=visible, hoverinfo='name',
    )

## Combined interactive view

Input model (gray) + fitted proxy (orange) + centerline (red) + rings + endpoints.
Toggle any layer via the legend.

In [ ]:
def combined_figure(data, height=750):
    """Build the combined input-vs-proxy figure for one loaded part dict.

    Branches on ``proxy_type``: swept tubes show centerline + rings +
    endpoints; surfaces of revolution show the rotation axis + profile r(h)
    + per-slice rings.
    """
    d = data
    f = go.Figure()
    # --- Original fitted-against model ---
    f.add_trace(mesh_trace(d['input_mesh'], 'input wrapped mesh', 'lightgray', opacity=0.35))
    f.add_trace(points_trace(d['sampled_pts'], 'sampled points', 'steelblue',
                             size=1.5, visible='legendonly'))
    # --- Fitted proxy result ---
    f.add_trace(mesh_trace(d['proxy_mesh'], 'fitted proxy surface', 'orange', opacity=0.55))
    if d['proxy_type'] == 'surface_of_revolution':
        f.add_trace(polyline_trace(d['axis_segs'], 'rotation axis', 'red', width=6))
        f.add_trace(polyline_trace(d['profile_segs'], 'profile r(h)', 'purple', width=5))
        f.add_trace(polyline_trace(d['ring_segs'], 'profile slices', 'green',
                                   width=2, close=True))
    else:
        f.add_trace(polyline_trace(d['centerline_segs'], 'centerline', 'red', width=6))
        f.add_trace(polyline_trace(d['ring_segs'], 'cross-section rings', 'green', width=3))
        f.add_trace(endpoints_trace(d['ep_a'], d['ep_b']))
    cov = d['params']['fitting_metrics'].get('coverage_ratio')
    cov_s = f' (coverage={cov:.2f})' if isinstance(cov, (int, float)) else ''
    f.update_layout(
        title=f"{d['part']} [{d['proxy_type']}]: input model vs fitted proxy{cov_s}",
        scene=dict(aspectmode='data', xaxis_title='x', yaxis_title='y', zaxis_title='z'),
        legend=dict(itemsizing='constant'),
        height=height, margin=dict(l=0, r=0, t=40, b=0),
    )
    return f


combined_figure(data).show()


## Side-by-side: input vs fitted

Two synchronized 3D panels — input wrapped model on the left, fitted proxy on the right.

In [ ]:
from plotly.subplots import make_subplots

_is_rev = data['proxy_type'] == 'surface_of_revolution'
fig2 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Input alpha-wrapped model',
                    f"Fitted proxy ({data['proxy_type']})"),
)

# Left: input.
fig2.add_trace(mesh_trace(input_mesh, 'input wrapped mesh', 'lightgray', opacity=0.5),
               row=1, col=1)

# Right: fitted result (layers depend on proxy type).
fig2.add_trace(mesh_trace(proxy_mesh, 'fitted proxy', 'orange', opacity=0.55), row=1, col=2)
if _is_rev:
    fig2.add_trace(polyline_trace(data['axis_segs'], 'rotation axis', 'red', width=6),
                   row=1, col=2)
    fig2.add_trace(polyline_trace(data['profile_segs'], 'profile r(h)', 'purple', width=5),
                   row=1, col=2)
    fig2.add_trace(polyline_trace(data['ring_segs'], 'profile slices', 'green', width=2,
                                  close=True), row=1, col=2)
else:
    fig2.add_trace(polyline_trace(centerline_segs, 'centerline', 'red', width=6), row=1, col=2)
    fig2.add_trace(polyline_trace(ring_segs, 'rings', 'green', width=3), row=1, col=2)
    fig2.add_trace(endpoints_trace(ep_a, ep_b), row=1, col=2)

fig2.update_layout(
    height=650, margin=dict(l=0, r=0, t=40, b=0), showlegend=True,
    scene=dict(aspectmode='data'), scene2=dict(aspectmode='data'),
)
fig2.show()


## Cross-section rings colored by reliability

Green = reliable section, red = unreliable. Useful to spot end-cap / low-coverage sections.

In [ ]:
import plotly.colors as pc


def reliability_figure(data, height=750):
    """Rings colored by reliability (green=reliable) for one loaded part dict.

    For swept tubes these are the fitted cross-sections; for surfaces of
    revolution they are the per-slice profile rings.
    """
    d = data
    is_rev = d['proxy_type'] == 'surface_of_revolution'
    f = go.Figure()
    f.add_trace(points_trace(d['sampled_pts'], 'sampled points', 'lightgray', size=1.2))
    base_segs = d['axis_segs'] if is_rev else d['centerline_segs']
    base_name = 'rotation axis' if is_rev else 'centerline'
    f.add_trace(polyline_trace(base_segs, base_name, 'black', width=4))

    rings = d['ring_segs']
    rel = d['section_reliability']
    n = min(len(rings), len(rel))
    for idx in range(n):
        ring = np.asarray(rings[idx])
        if len(ring) > 1:
            ring = np.vstack([ring, ring[:1]])
        r = float(rel[idx])
        color = pc.sample_colorscale('RdYlGn', r)[0]
        label = 'slice' if is_rev else 'section'
        f.add_trace(go.Scatter3d(
            x=ring[:, 0], y=ring[:, 1], z=ring[:, 2], mode='lines',
            line=dict(color=color, width=4),
            name=f'{label} {idx} (r={r:.2f})', showlegend=False,
            hovertext=f'{label} {idx}: reliability={r:.2f}', hoverinfo='text',
        ))

    if d['ep_a'] is not None:
        f.add_trace(endpoints_trace(d['ep_a'], d['ep_b']))
    f.update_layout(
        title=f"{d['part']} [{d['proxy_type']}]: rings colored by reliability (green=reliable)",
        scene=dict(aspectmode='data'), height=height,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return f


reliability_figure(data).show()


In [ ]:
# Quick metrics summary (keys vary by proxy type).
m = params['fitting_metrics']
print('proxy_type:', data['proxy_type'])
for k in ['coverage_ratio', 'chamfer', 'mean_input_to_proxy', 'mean_proxy_to_input',
          'n_sections', 'pct_unreliable', 'mean_reliability', 'radius_mean',
          'centerline_length', 'n_profile_slices', 'axis_length',
          'mean_angular_coverage', 'mean_radial_deviation']:
    if k in m:
        print(f'{k:24s}: {m[k]}')


# All parts — gallery

The cells above focus on the part chosen via `PART`. The cells below render the
**same visualization for every fitted part** found under `output/`, so you can
inspect them all at once. Each part uses its auto-selected proxy type:
tube-like parts (`spout`, `handle`) as swept generalized cylinders, round parts
(`teapot_body`, `lid`, `knob`) as surfaces of revolution.


In [ ]:
# Metrics overview table for all parts.
import pandas as pd

rows = []
for part in available_parts:
    try:
        with open(os.path.join(OUTPUT_ROOT, part, 'params.json')) as f:
            prm = json.load(f)
    except FileNotFoundError:
        continue
    mm = prm['fitting_metrics']
    rows.append({
        'part': part,
        'proxy_type': prm.get('proxy_type', 'generalized_cylinder'),
        'coverage': mm.get('coverage_ratio'),
        'chamfer': mm.get('chamfer'),
        'mean_reliability': mm.get('mean_reliability'),
        'radius_mean': mm.get('radius_mean'),
        # swept-tube only / revolve only metrics (NaN where not applicable):
        'pct_unreliable': mm.get('pct_unreliable'),
        'centerline_length': mm.get('centerline_length'),
        'axis_length': mm.get('axis_length'),
    })

overview = pd.DataFrame(rows).sort_values('coverage', ascending=False)
overview.reset_index(drop=True)


In [ ]:
# Combined input-vs-proxy view for EVERY part.
all_data = {}
for part in available_parts:
    print('Loading', part, '...')
    all_data[part] = load_part(part)

for part in available_parts:
    combined_figure(all_data[part], height=600).show()

In [ ]:
# Reliability (rings) view for EVERY part that has rings.
for part in available_parts:
    d = all_data[part]
    if len(d['ring_segs']) == 0:
        print(f'{part}: no rings (skipped)')
        continue
    reliability_figure(d, height=600).show()
